# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data that `01-data_collection.ipynb` saved to `data/raw/` and cleans it.

The main output is the cleaned FracTracker data centers table with an `is_crypto` label.

The EIA sections near the end are a first look only. They are not saved yet.

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import geocoder
from geopy.distance import geodesic

In [2]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [3]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1719, 53)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup
779,3969043,94b7512c-4c85-4870-a9b6-3ae3ccd1911b,425,Stargate Data Center,2369 Ellsworth Bailey Rd,Warren,OH,44481.0,Trumbull,41.153240,-80.880300,Proposed,High,Google Maps and other sources,NaN,Bristolville 25 Developer LLC,NaN,1500,1500.0,1500.0,"Mega campus (>1,000 MW)",6,"Grid (unspecified mix), Biomass",NaN,NaN,NaN,NaN,NaN,NaN,NaN,$3.6 billion,NaN,Yes,https://www.wkbn.com/news/local-news/lordstown...,Emerging Concern,NaN,NaN,NaN,NaN,Former General Motors plant outside Lordstown,KE/MG,Media Monitoring,https://www.chron.com/business/article/openai-...,https://www.wfmj.com/story/53109325/lordstown-...,https://www.wfmj.com/story/53113377/data-cente...,https://www.wfmj.com/story/53208675/lordstown-...,NaN,NaN,NaN,NaN,1.758845e+12,1.774224e+12,NaN
134,3968398,f5b15818-571f-4407-8402-cd1738782b36,58,Alpharetta GA1,11650 Great Oaks Way,Alpharetta,GA,30022.0,Fulton,34.063183,-84.272896,Operating,High,Other sources,NaN,365 Data Centers,NaN,4,4.0,4.0,Small (0-10 MW),2,NaN,NaN,NaN,NaN,NaN,NaN,77300.0,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,Sci4GA,https://baxtel.com/data-center/sungard-atlanta...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.760573e+12,1.760573e+12,NaN
13,3968277,3112837a-6cf4-41dd-9417-93315b839460,1792194306849-4882,US Space Command Data Center,Redstone Arsenal Army Base,Redstone Arsenal,AL,35808.0,Madison,34.648544,-86.655075,Proposed,Medium,Google Maps and other sources,NaN,US Department of War,NaN,NaN,NaN,NaN,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,700000.0,64.0,$4 billion,2031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.datacenterdynamics.com/en/news/us-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.792109e+12,1.792109e+12,NaN


In [4]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1719.0,NaN,NaN,NaN,3969123.0,496.376873,3968264.0,3968693.5,3969123.0,3969552.5,3969982.0
GlobalID,1719,1719,f4107c30-66df-4c1e-8022-28e6c8fe2b5d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1719,1719,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1719,1592,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1674,1662,Farm-to-Market 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1715,847,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1719,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1706.0,NaN,NaN,NaN,43959.849941,26477.182681,1085.0,20166.0,30349.0,72984.5,99734.0
county,1716,481,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1719.0,NaN,NaN,NaN,37.583959,4.23384,21.480452,33.777525,38.746647,40.058871,70.18478


### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `sizerank` / `sizerank_numeric` | Mismatched values that are not consistent across categories. | Align sizerank wiht sizerank_numeric column labels. |
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`sizerank` / `sizerank_numeric`

In [5]:
# Mappings
size_rank_mw = {
   'Unknown': 1,
   'Small (0-10 MW)': 2,
   'Medium (11-50 MW)': 3,
   'Large (51-99 MW)': 4,
   'Hyperscale (100-999 MW)': 5,
   'Mega campus (>1,000 MW)': 6
}

# Fix sizerank labels
data_centers['sizerank'] = data_centers['sizerank'].replace('Mega Campus (>1,000 MW)', 'Mega campus (>1,000 MW)')
data_centers['sizerank_numeric'] = data_centers['sizerank'].apply(lambda x: size_rank_mw[x])

`location_confidence`

In [6]:
# Small fix for location confidence values
data_centers['location_confidence'] = (
    data_centers['location_confidence']
    .apply(lambda x: 'High' if x == 'high' else x)
)

`sizerank`

In [7]:
# The Mega campus category is spelled two ways
data_centers["sizerank"] = data_centers["sizerank"].replace(
    "Mega Campus (>1,000 MW)", "Mega campus (>1,000 MW)"
)

data_centers["sizerank"].value_counts(dropna=False)

sizerank
Unknown                    891
Hyperscale (100-999 MW)    402
Medium (11-50 MW)          140
Mega campus (>1,000 MW)    126
Small (0-10 MW)            118
Large (51-99 MW)            42
Name: count, dtype: int64

`mw`

In [8]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
6,210,210.0,210.0


`power_source`

In [9]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
119,Oil/Diesel,0,0,0,0,0,0,0,0,0,0,1,0,0,0
639,"Solar, Grid (unspecified mix)",0,0,0,0,0,1,0,0,0,0,0,0,1,0
1045,Solar,0,0,0,0,0,0,0,0,0,0,0,0,1,0
10,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
617,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0


`number_of_buildings`

In [10]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
10,4,4.0,4.0
11,2,2.0,2.0
15,14,14.0,14.0
16,4,4.0,4.0


`cooling_source` + `cooling_type`

In [11]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
913,Water,0,1,0
62,Hybrid air/water,0,0,1
1093,Hybrid air/water,0,0,1
1154,Hybrid air/water,0,0,1
772,Water,0,1,0


In [12]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
1703,Closed loop,1,0,0
1691,Closed loop,1,0,0
532,Open loop,0,0,1
1180,Closed loop,1,0,0
471,Closed loop,1,0,0


`project_cost`

In [14]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

ValueError: could not convert string to float: '471.8(kati2)'

### FracTracker Crypto Mining Operations

In [15]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

Rows and columns: (302, 18)


,Name,City,State,Address,Company,Probable power source,Links,Status,lat,long,Precise Location?,MapCat,GEM_links,Other updates,Date,MW (NYTimes),Unnamed: 16,Unnamed: 17
192,Longhorn A well pad (crypto operation closed),Horton Township,PA,"Horton, PA",Diversified Production LLC,NaN,https://www.newsfromthestates.com/article/rura...,Unknown,41.318333,-78.765194,Yes,Shut down/ moved,NaN,NaN,5/4/2025,NaN,NaN,NaN
198,Standard Power Beaver County facility,Shippingport,PA,"Rt 168, Shippingport, PA 15077",Standard Power,BTM/Nuclear,https://www.datacenterdynamics.com/en/news/sta...,Unknown,40.633840,-80.419530,Yes,Current status unknown,https://www.gem.wiki/Standard_Power_Beaver_Cou...,NaN,NaN,NaN,NaN,NaN
268,Brazos Valley Mine,College Station,TX,NaN,White Rock Management,Flared gas,https://www.prnewswire.com/news-releases/white...,Operating,30.593955,-96.301274,No,Operating / under construction,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
crypto.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,302,250,CleanSpark,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,283,235,Dalton,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,302,41,GA,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,194,189,"430 Davis Rd, Vidalia, GA 30474",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Company,283,174,CleanSpark,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Probable power source,163,36,Unknown,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Links,299,263,https://www.wbir.com/article/tech/east-tenness...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,301,11,Operating,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,302.0,NaN,NaN,NaN,37.830255,5.191345,25.454595,34.456008,36.356141,41.261212,69.71761
long,302.0,NaN,NaN,NaN,-128.525789,651.005258,-11402.57827,-98.202512,-88.774187,-82.477328,-69.170992


### FracTracker Crypto Mining Operations - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `Probable power source` | Power source values use inconsistent naming conventions. | Normalize them to the FracTracker data center categories using keyword mapping. |
| `Precise Location?` | Values represented as 'Yes' / 'No' strings. | Update to a boolean representation. |
| `MW (NYTimes)` | A range of values including '100 MW', '100 MW+', '100 MW --> 200 MW'. | Standardize to `mw_low` and `mw_high` to keep consistent representation in both datasets. |

`Probable power source`

In [17]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    source = source.lower()
    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "battery": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)

    # Grid
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    # Grid + another identified source = hybrid
    non_grid_sources = found_sources - {"grid_unspecified_mix"}

    if "grid_unspecified_mix" in found_sources and non_grid_sources:
        found_sources.add("hybrid_onsite_and_grid")

    return sorted(found_sources)


crypto["norm_power_source"] = crypto["Probable power source"].apply(normalize_sources)
crypto.loc[crypto["norm_power_source"].str.len() > 0][["norm_power_source"]].sample(5)

,norm_power_source
283,[hydroelectric]
265,[natural_gas]
39,[grid_unspecified_mix]
289,[coal]
87,[coal]


`Precise Location?`

In [18]:
# Change precise location to a boolean
crypto["Precise Location?"] = crypto["Precise Location?"].apply(lambda x: x == 'Yes')
crypto[['Precise Location?']].sample(5)

,Precise Location?
175,True
297,False
56,False
267,False
219,False


`MW (NYTimes)`

In [19]:
# Function for standardizing mw values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Fix typo
    text = text.replace("nw", "mw")

    # Extract numbers followed by MW or GW
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Add new features to handle ranges
crypto[["mw_low", "mw_high"]] = pd.DataFrame(
    crypto["MW (NYTimes)"].apply(standardize_mw).tolist(),
    index=crypto.index
)

crypto.loc[crypto['MW (NYTimes)'].notna()][["MW (NYTimes)", "mw_low", "mw_high"]].sample(5)

,MW (NYTimes),mw_low,mw_high
53,100 MW,100.0,100.0
103,60 MW-->150 MW,60.0,150.0
248,700 MW,700.0,700.0
69,16 MW,16.0,16.0
153,7.5 MW,7.5,7.5


### Address Standardization & Comparison 

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. These values are separated so the final location fields are: 

- **Crypto**: `Address`, `City`, `State`, `Zip` 
- **Data Centers**: `address`, `city`, `state`, `zip`

#### Address Abbreviations

In [20]:
ADDRESS_ABBREVIATIONS = {
    # Directions
    'north': 'n',
    'south': 's',
    'east': 'e',
    'west': 'w',
    'northeast': 'ne',
    'northwest': 'nw',
    'southeast': 'se',
    'southwest': 'sw',

    # Street suffixes
    'street': 'st',
    'avenue': 'ave',
    'road': 'rd',
    'drive': 'dr',
    'lane': 'ln',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'place': 'pl',
    'parkway': 'pkwy',
    'highway': 'hwy',
    'trail': 'trl',
    'terrace': 'ter',
    'square': 'sq',
    'crossing': 'xing',
    'center': 'ctr',
    'centre': 'ctr',
    'point': 'pt',
    'ridge': 'rdg',
    'run': 'run',
    'view': 'vw',
    'village': 'vlg',
    'station': 'sta',
    'landing': 'lndg',
    'junction': 'jct',
    'extension': 'ext',
    'expressway': 'expy',
    'freeway': 'fwy',
    'turnpike': 'tpke',
    'causeway': 'cswy',
    'crescent': 'cres',
    'cove': 'cv',
    'fork': 'frk',
    'garden': 'gdn',
    'gardens': 'gdns',
    'grove': 'grv',
    'harbor': 'hbr',
    'harbour': 'hbr',
    'heights': 'hts',
    'hill': 'hl',
    'hills': 'hls',
    'island': 'is',
    'islands': 'iss',
    'knoll': 'knl',
    'knolls': 'knls',
    'lake': 'lk',
    'lakes': 'lks',
    'manor': 'mnr',
    'manors': 'mnrs',
    'meadow': 'mdw',
    'meadows': 'mdws',
    'mount': 'mt',
    'mountain': 'mtn',
    'orchard': 'orch',
    'pine': 'pne',
    'pines': 'pnes',
    'plaza': 'plz',
    'port': 'prt',
    'prairie': 'pr',
    'radial': 'radl',
    'river': 'riv',
    'shore': 'shr',
    'shores': 'shrs',
    'spring': 'spg',
    'springs': 'spgs',
    'summit': 'smt',
    'union': 'un',
    'valley': 'vly',
    'valleys': 'vlys',
    'well': 'wl',
    'wells': 'wls',
    
    # Unit Overrides
    'apartment': 'apt',
    'apt.': 'apt',
    'suite': 'ste',
    'ste.': 'ste',
    'unit': 'unit',
    '#': 'unit',
    'building': 'bldg',
    'floor': 'fl',
    'room': 'rm',
    'department': 'dept'
}

#### Helper Functions

These functions are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [21]:
def standardize_address(address):
    if pd.isna(address):
        return np.nan
    
    address = address.lower().strip()
    
    # Remove special characters
    address = re.sub(r'[^a-z0-9\s]', '', address)
    address_parts = address.split()
    
    # Update addresses using abbreviations
    cleaned_parts = [
        ADDRESS_ABBREVIATIONS.get(part, part)
        for part in address_parts
    ]
    
    return " ".join(cleaned_parts)


def remove_city_state(address, city, state):
    if pd.isna(address):
        return np.nan

    address = str(address)

    # Remove ZIP from address
    address = re.sub(r'\s*,?\s*\d{5}\s*$', '', address)

    # Remove State from address
    if pd.notna(state):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(state))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    # Remove City from address
    if pd.notna(city):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(city))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    return address.strip(' ,')

In [22]:
# Extract 5-digit zip if exists
crypto['Zip'] = crypto['Address'].str.extract(r'(\d{5})\s*$')[0]
crypto['Zip'] = crypto['Zip'].replace('', np.nan).astype('Int64') # missing zip

# Remove city, state, and zip from addresses in crypto
crypto['Address'] = crypto.apply(
    lambda row: remove_city_state(
        row['Address'],
        row['City'],
        row['State']
    ),
    axis=1
)

# Applying the transformations to the addresses in both datasets
crypto['Address'] = crypto['Address'].apply(standardize_address)
data_centers['address'] = data_centers['address'].apply(standardize_address)


# data_centers[['address', 'city', 'state', 'zip']].sample(5)
crypto[['Address', 'City', 'State', 'Zip']].sample(5)

,Address,City,State,Zip
301,campstool business park 6106 yellowstone rd,Cheyenne,WY,82009
273,bluestone business and technology park 326 e s...,Tazewell,VA,24630
271,uinta basin,NaN,UT,<NA>
185,4800 s 24th st w muskogee,Port Muskogee,OK,74401
109,NaN,Waverly,KY,<NA>


#### Address Comparison

The standardized datasets are merged using exact matches on address, city, and state to identify facilities that appear in both datasets. This allows us to identify data centers associated with known crypto mining locations, providing more information for later analysis. Using exact address matching, 19 matching records were identified between the two datasets.

In [23]:
# Find matching addresses between the datasets
matches = crypto[crypto['Address'].notna()].merge(
    data_centers[data_centers['address'].notna()],
    left_on=['Address', 'City', 'State'],
    right_on=['address', 'city', 'state'],
    how='inner',
    suffixes=('_crypto', '_dc')
)

print(f"{len(matches)} matches found between the crypto and data centers datasets.")
matches[['facility_name', 'Name', 'lat_crypto', 'long_crypto', 'lat_dc', 'long_dc', 'address', 'city', 'state']]

19 matches found between the crypto and data centers datasets.


,facility_name,Name,lat_crypto,long_crypto,lat_dc,long_dc,address,city,state
0,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific Whitfield County facility (sel...,34.713160,-84.951430,34.713160,-84.951430,2205 industrial s rd,Dalton,GA
1,Riot 1,Block Mining McCracken County facility,37.088310,-88.703050,37.088310,-88.703050,5600 commerce dr,Paducah,KY
2,Riot 2,Riot Platforms greenfield site,37.088592,-88.676158,37.088593,-88.676155,coleman rd,Paducah,KY
3,Hyperscale Data Michigan Data Center,Michigan Data Center,41.993228,-86.098603,41.992455,-86.098976,415 e pr ronde st,Dowagiac,MI
4,CoreWeave/ Applied Digital,Applied Digital Stutsman County facility,47.027383,-98.679666,47.027080,-98.679600,2671 hwy 20 se,Jamestown,ND
5,Applied Digital: Polaris Forge 1,"Applied Digital, being repurposed from crypto ...",46.010438,-98.570800,46.012590,-98.570700,9685 87th ave se,Ellendale,ND
6,Vulcan Infrastructure and Power Inc (formerly ...,Greenidge Generation Yates County facility,42.676870,-76.944910,42.676870,-76.944910,590 plant rd,Dresden,NY
7,BlockFusion aka North East Data Center LLC,North East Data LLC Niagara facility,43.082864,-79.003495,43.083187,-79.003170,5380 frontier ave,Niagara Falls,NY
8,9Lives Data Center,Nine Lives Digital Group,43.123208,-79.040614,43.123890,-79.041180,3807 highland ave,Niagara Falls,NY
9,North Tonawanda Data Center,Digipower X Fortistar Niagara County facility,43.060860,-78.853760,43.060860,-78.853760,1070 erie ave,North Tonawanda,NY


#### Crypto Label for Data Centers

The FracTracker data center list does not say which facilities are crypto mines. We build an `is_crypto` label here so `03-supervised-learning.ipynb` can load it from `data/processed/`.

The label comes from **two different sources**:

1. **The FracTracker crypto dataset.** A data center is crypto if it matches a known crypto mining site, either:
   - by location: a crypto site is within `MAX_MILES` of it, or
   - by exact address (the Address Comparison above).
2. **The FracTracker data centers dataset itself.** Some data centers already say they are crypto in their `purpose` or `facility_name` (for example "Bitcoin" or "Crypto mining/AI"). Many of these are not in the crypto dataset at all.

We keep one column for each check (`crypto_from_distance`, `crypto_from_address`, `crypto_from_dc_dataset`) so we can always see where a label came from.

##### How old is each dataset?

The crypto dataset is older than the data centers dataset:

- **Crypto sites:** the `Date` column runs from March 2023 to April 2026, but most entries are from 2023 and 2024 (median: May 2024).
- **Data centers:** the `date_updated` column runs from January 2025 to September 2026 (median: November 2025).

So there is roughly a year and a half between the typical crypto entry and the typical data center entry. This matters for the crypto label:

- A data center that matched a crypto site tells us the site **was** crypto when the crypto dataset was made. It may not still be crypto. Several sites are converting from crypto mining to AI (for example Applied Digital, Crexi and Keel Infrastructure).
- A "Proposed" crypto site from 2023 may never have been built, or may have been built as a different kind of data center.
- The `crypto_from_dc_dataset` column comes from the newer dataset, so it is the more up-to-date of the two sources.

In [24]:
# Compare how old each dataset is
crypto_dates = pd.to_datetime(crypto["Date"], format="%m/%d/%Y", errors="coerce")
dc_dates = pd.to_datetime(data_centers["date_updated"], unit="ms")

print("Crypto sites with a date:", crypto_dates.notna().sum(), "of", len(crypto))
print("Crypto dates:            ", crypto_dates.min().date(), "to", crypto_dates.max().date())
print("Crypto median date:      ", crypto_dates.median().date())
print()
print("Data centers last updated:", dc_dates.min().date(), "to", dc_dates.max().date())
print("Data centers median date: ", dc_dates.median().date())
print()
print("Crypto sites by year:")
print(crypto_dates.dt.year.value_counts().sort_index())
print()
print("Data centers by year last updated:")
print(dc_dates.dt.year.value_counts().sort_index())

Crypto sites with a date: 167 of 302
Crypto dates:             2023-03-02 to 2026-04-15
Crypto median date:       2024-05-17

Data centers last updated: 2025-01-04 to 2026-10-16
Data centers median date:  2025-11-24

Crypto sites by year:
Date
2023.0    39
2024.0    96
2025.0    30
2026.0     2
Name: count, dtype: int64

Data centers by year last updated:
date_updated
2025.0    892
2026.0    824
Name: count, dtype: int64


##### Drop invalid coordinates

A few rows have coordinates that cannot be real: a longitude of `-11402.57827` (likely a typo) and a data center at `0, 0` (missing location). We can't measure distance for these, so we drop them.

In [25]:
def valid_coordinates(df):
    return df["lat"].between(-90, 90) & df["long"].between(-180, 180) & ~((df["lat"] == 0) & (df["long"] == 0))

print("Dropping data centers:", data_centers.loc[~valid_coordinates(data_centers), "facility_name"].tolist())
print("Dropping crypto sites:", crypto.loc[~valid_coordinates(crypto), "Name"].tolist())

data_centers = data_centers[valid_coordinates(data_centers)].reset_index(drop=True)
crypto = crypto[valid_coordinates(crypto)].reset_index(drop=True)

Dropping data centers: []
Dropping crypto sites: ['Cipher "Chief Facility"']


##### Distance between every crypto site and every data center

`geodesic` from the `geopy` package takes two `(lat, long)` points and returns the distance between them along the Earth's surface.

We loop over every crypto site and every data center, which is about 500,000 pairs, so this cell takes around 30 seconds.

The result is a table with one row per crypto site and one column per data center.

In [26]:
crypto_points = list(zip(crypto["lat"], crypto["long"]))
dc_points = list(zip(data_centers["lat"], data_centers["long"]))

distances = []
for crypto_point in crypto_points:
    row = []
    for dc_point in dc_points:
        row.append(geodesic(crypto_point, dc_point).miles)
    distances.append(row)

# turn the list of rows into a table numpy can search
distances = np.array(distances)

print("Distance table (crypto sites, data centers):", distances.shape)

Distance table (crypto sites, data centers): (301, 1719)


##### Nearest data center for each crypto site

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [27]:
nearest_dc = distances.argmin(axis=1)

crypto["nearest_dc_miles"] = distances.min(axis=1)
crypto["nearest_dc_name"] = data_centers["facility_name"].values[nearest_dc]
crypto["nearest_dc_operator"] = data_centers["operator_name"].values[nearest_dc]

for miles in [0.1, 0.25, 0.5, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = crypto[crypto["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Crypto sites with a data center within 0.1 mi: 23
Crypto sites with a data center within 0.25 mi: 32
Crypto sites with a data center within 0.5 mi: 41
Crypto sites with a data center within 1 mi: 48
Crypto sites with a data center within 2 mi: 58
Crypto sites with a data center within 5 mi: 75


Most crypto sites have no data center nearby. Many crypto mines are simply not in the data center list.

The table below puts the names side by side so we can check whether close pairs are really the same facility.

In [28]:
columns = [
    "Name", "Company", "Precise Location?",
    "nearest_dc_name", "nearest_dc_operator", "nearest_dc_miles"
]

crypto.loc[crypto["nearest_dc_miles"] <= 2, columns].sort_values("nearest_dc_miles")

,Name,Company,Precise Location?,nearest_dc_name,nearest_dc_operator,nearest_dc_miles
55,Core Scientific Whitfield County facility (sel...,CleanSpark,True,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific,0.000000
98,Block Mining McCracken County facility,Riot,True,Riot 1,Riot,0.000000
173,Greenidge Generation Yates County facility,Greenidge Generation LLC,True,Vulcan Infrastructure and Power Inc (formerly ...,NaN,0.000000
178,Digipower X Fortistar Niagara County facility,Digipower X,True,North Tonawanda Data Center,NaN,0.000000
194,Bitfarm (Stronghold) Panther Creek Plant,Bitfarm,True,Keel Infrastructure Panther Creek,Keel Infrastructure,0.000000
195,Bitfarm (Stronghold) Scrubgrass Plant,Bitfarm,True,Keel Infrastructure Scrubgrass,Keel Infrastructure,0.000000
202,BitFarms Mercer County facility,BitFarms,True,Sharon Data Center,NaN,0.000061
260,Riot Platforms,Riot Platforms,True,Riot Platforms,NaN,0.000083
154,Atlas Power Williams County facility,FX Solutions > Atlas Power,True,Atlas Power,NaN,0.000139
103,Riot Platforms greenfield site,Riot,False,Riot 2,Riot,0.000171


##### Source 1: the crypto dataset

We label from the data center side: for each data center, find the closest crypto site. If it is within `MAX_MILES`, `crypto_from_distance` is True.

Labeling this way handles two things:
- The crypto file repeats some sites (same name or same coordinates), and several crypto rows can point to one data center.
- Every data center gets exactly one label.

`0.5` miles is a starting point. Some sites under 2 miles are clearly different facilities, for example CleanSpark (Clinton) and Amazon Clinton Data Center.

We also keep the exact address matches from above. Two of them are real matches that are a little more than 0.5 miles apart (Helios Data Center and Riot Platforms Rockdale).

In [29]:
MAX_MILES = 0.5

nearest_crypto = distances.argmin(axis=0)

data_centers["crypto_miles"] = distances.min(axis=0)
data_centers["crypto_name"] = crypto["Name"].values[nearest_crypto]
data_centers["crypto_company"] = crypto["Company"].values[nearest_crypto]
data_centers["crypto_status"] = crypto["Status"].values[nearest_crypto]

# Check 1: a crypto site is within MAX_MILES of the data center
data_centers["crypto_from_distance"] = data_centers["crypto_miles"] <= MAX_MILES

# Check 2: the data center matched a crypto site by exact address
address_match_ids = list(matches["facility_id"])
data_centers["crypto_from_address"] = data_centers["facility_id"].isin(address_match_ids)

print("Crypto from distance:", data_centers["crypto_from_distance"].sum())
print("Crypto from address: ", data_centers["crypto_from_address"].sum())

Crypto from distance: 41
Crypto from address:  19


##### Source 2: the data centers dataset

We search `purpose` and `facility_name` for words that mean crypto mining. We do not search for "crypto" everywhere, because it would also match "NSA Texas Cryptology Center".

In [30]:
# "^crypto" means the text starts with "crypto"
crypto_words = "bitcoin|crypto mining|crypto mine|cryptomine|cryptocurrency|^crypto"

purpose_says_crypto = data_centers["purpose"].str.contains(crypto_words, case=False, na=False)
name_says_crypto = data_centers["facility_name"].str.contains(crypto_words, case=False, na=False)

# Check 3: the data center says it is crypto
data_centers["crypto_from_dc_dataset"] = purpose_says_crypto | name_says_crypto

print("Crypto from purpose or name:", data_centers["crypto_from_dc_dataset"].sum())

Crypto from purpose or name: 21


##### Final label

A data center gets `is_crypto = 1` if any of the three checks is True.

In [31]:
matched_crypto_site = data_centers["crypto_from_distance"] | data_centers["crypto_from_address"]

data_centers["is_crypto"] = (
    matched_crypto_site | data_centers["crypto_from_dc_dataset"]
).astype(int)

# only keep crypto site details for data centers that matched a crypto site
data_centers.loc[~matched_crypto_site, ["crypto_name", "crypto_company", "crypto_status"]] = np.nan

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

Data centers labeled crypto: 59 of 1719
Share labeled crypto: 3.4 %


In [32]:
columns = [
    "facility_name", "operator_name", "state", "purpose",
    "crypto_from_distance", "crypto_from_address", "crypto_from_dc_dataset",
    "crypto_name", "crypto_miles"
]

data_centers.loc[data_centers["is_crypto"] == 1, columns].sort_values("crypto_miles")

,facility_name,operator_name,state,purpose,crypto_from_distance,crypto_from_address,crypto_from_dc_dataset,crypto_name,crypto_miles
219,Core Scientific Dalton 1 & 2 Data Center Campus,Core Scientific,GA,NaN,True,True,False,Core Scientific Whitfield County facility (sel...,0.000000
457,Riot 1,Riot,KY,NaN,True,True,False,Block Mining McCracken County facility,0.000000
689,Vulcan Infrastructure and Power Inc (formerly ...,NaN,NY,Bitcoin transitioning to AI,True,True,True,Greenidge Generation Yates County facility,0.000000
707,North Tonawanda Data Center,NaN,NY,NaN,True,True,False,Digipower X Fortistar Niagara County facility,0.000000
861,Keel Infrastructure Scrubgrass,Keel Infrastructure,PA,NaN,True,True,False,Bitfarm (Stronghold) Scrubgrass Plant,0.000000
875,Keel Infrastructure Panther Creek,Keel Infrastructure,PA,AI,True,True,False,Bitfarm (Stronghold) Panther Creek Plant,0.000000
909,Sharon Data Center,NaN,PA,NaN,True,True,False,BitFarms Mercer County facility,0.000061
989,Riot Platforms,NaN,TX,NaN,True,False,False,Riot Platforms,0.000083
625,Atlas Power,NaN,ND,Crypto,True,False,True,Atlas Power Williams County facility,0.000139
458,Riot 2,Riot,KY,NaN,True,True,False,Riot Platforms greenfield site,0.000171


##### How much does the cutoff matter?

The number of data centers matched by distance depends on `MAX_MILES`. This helps us choose it. (This table only counts the distance check, not the address or `purpose`/`facility_name` checks.)

In [33]:
rows = []
for miles in [0.1, 0.25, 0.5, 1, 2]:
    labeled_dcs = data_centers[data_centers["crypto_miles"] <= miles]
    matched_crypto = crypto[crypto["nearest_dc_miles"] <= miles]

    rows.append({
        "max_miles": miles,
        "data_centers_labeled": len(labeled_dcs),
        "crypto_sites_matched": len(matched_crypto),
    })

pd.DataFrame(rows)

,max_miles,data_centers_labeled,crypto_sites_matched
0,0.10,23,23
1,0.25,32,32
2,0.50,41,41
3,1.00,51,48
4,2.00,65,58


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [34]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
4714,11479,Madison Gas & Electric Co,56386,Top of Iowa Windfarm III,570 410th St,Kensett,IA,50448,Worth,43.3597,-93.3208,MRO,MISO,Midcontinent Independent Transmission System O...,NaN,22,RE,1,Electric Utility,N,NaN,N,NaN,N,NaN,N,X,NaN,Interstate Power and Light Co,9417,IA,161,,,N,NaN,NaN,NaN,NaN,NaN,N,N
6771,58649,"SunStrong Management, LLC",58889,ASTI,26150 Asti Rd,Cloverdale,CA,95427,Sonoma,38.761389,-122.975278,WECC,CISO,California Independent System Operator,NaN,22,NR,2,IPP Non-CHP,N,NaN,N,NaN,N,NaN,N,X,NaN,Pacific Gas & Electric Co,14328,CA,480,,,N,NaN,NaN,NaN,NaN,NaN,N,N
8557,60584,Onyx Asset Services Group,60967,Onyx - Lamphear Road CSG,6216 Lamphear Road,Rome,NY,13440,Oneida,43.18413,-75.476724,NPCC,NYIS,New York Independent System Operator,NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,16-1132,N,NaN,N,NaN,NaN,New York Power Authority,15296,NaN,13.2,,,N,NaN,NaN,NaN,NaN,NaN,N,N


In [35]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [36]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
5877,55951,Constellation Nuclear,6026.0,LaSalle Generating Station,IL,LaSalle,1,Nuclear,ST,NaN,S,X,X,86191101,86191101,1170.0,0.9,1130.5,1172.5,150,N,,,OP,X,1.0,1984.0,,,N,IPP Non-CHP,2.0,X,NUC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,OVER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,N,,,N,N,NaN
18841,57313,Tesla Inc.,60822.0,Intel - Ocotillo Campus Solar,AZ,Maricopa,PV2,Solar Photovoltaic,PV,NaN,S,X,X,NaN,NaN,1.4,,1.4,1.4,,N,,,OP,X,3.0,2018.0,,,N,IPP Non-CHP,2.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,NaN,NaN,NaN
22074,16604,City of San Antonio - (TX),63688.0,Topaz Generating,TX,Galveston,CTG-6,Natural Gas Fired Combustion Turbine,GT,NaN,W,X,X,138_ATT_TPZ,NaN,60.5,0.85,44.5,49.8,22.5,N,,,OP,X,7.0,2021.0,,,N,Electric Utility,1.0,X,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,10M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN


In [37]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Export data prior to failed cells

In [38]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

Saved: fractracker_data_centers_clean.csv with 1719 rows and 83 columns


EIA-860 Defining Energy Source Features

In [73]:
energy_sources = pd.read_csv(
    DATA_DIR / "eia860_2025" / "eia_860_energy_source_codes.csv",
)
energy_sources.rename(columns={'Energy Source Code': 'Energy Source 1'}, inplace=True)
energy_sources.sample(3)

,Energy Source 1,Description
28,OBG,"Other Biomass Gas (including digester gas, met..."
18,OG,Other Gas (specify in SCHEDULE 7)
26,WDL,Wood Waste Liquids excluding Black Liquor (inc...


In [96]:
generators['Energy Source 1'].unique()
generator_features = generators[['State','County','Energy Source 1','Technology','Time from Cold Shutdown to Full Load','Summer Capacity (MW)','Winter Capacity (MW)']].copy()
# Dropping description to create categorical feature from Energy Source
generator_features_full = pd.merge(generator_features, energy_sources, on='Energy Source 1', how='left').drop('Description', axis=1)
generator_features_full.rename(columns={'Category':'Energy Category','Time from Cold Shutdown to Full Load':'Time to Full Load'}, inplace=True)
# Clean Time to Full Load from string to numerical value
generator_features_full.sample(15)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW)
29202,FL,Duval,NG,Natural Gas Internal Combustion Engine,1H,2.5,2.5
18000,KY,Bracken,WAT,Conventional Hydroelectric,1H,35,35
19735,NC,Beaufort,SUN,Solar Photovoltaic,NaN,12.4,12.4
8684,MO,Ralls,WAT,Hydroelectric Pumped Storage,1H,31,31
21904,VA,Frederick,NG,Natural Gas Fired Combustion Turbine,1H,15,15
27396,VA,Henry,SUN,Solar Photovoltaic,NaN,20,20
13694,CA,Kern,NG,Natural Gas Fired Combustion Turbine,1H,2.7,3
334,AK,Dillingham,DFO,Petroleum Liquids,10M,2.2,2.2
28895,GA,Oglethorpe,SUN,Solar Photovoltaic,NaN,38,38
23135,MA,Plymouth,SUN,Solar Photovoltaic,NaN,2,2


Time to Full Load in Minutes Clean Up

In [97]:
def time_in_min(stringval):
    val = str(stringval).lower().strip()
    # Match digits versus text
    regex = re.match(r"(\d+)(\s*[a-z]+)", val)
    if not regex:
        return None

    if regex.group(2) in ['m']:
        return int(regex.group(1))
    elif regex.group(2) in ['h']:
        return int(regex.group(1))*60
    else:
        return None

In [98]:
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].apply(time_in_min)


In [99]:
generator_features_full.describe()
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].fillna(generator_features_full['Time to Full Load'].mean().round(1))

One-Hot Encoding for Energy

In [100]:
one_hot_energy = pd.get_dummies(generator_features_full['Technology']).astype(int)
generator_features_full = generator_features_full.join(one_hot_energy)

#convert from object to numeric for MW
generator_features_full[["Summer Capacity (MW)", "Winter Capacity (MW)"]] = generator_features_full[["Summer Capacity (MW)", "Winter Capacity (MW)"]].apply(
    pd.to_numeric,
    errors = "coerce"
)

In [102]:
generator_features_full.sample(5)

,State,County,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass
27859,CT,New Haven,NG,Other Natural Gas,140.7,2.2,2.2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
190,AL,Washington,NG,Natural Gas Fired Combined Cycle,720.0,380.0,421.0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
30964,NY,Steuben,SUN,Solar Photovoltaic,140.7,5.0,5.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
21307,CA,Kern,SUN,Solar Photovoltaic,140.7,20.0,20.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
30798,CA,San Luis Obispo,MWH,Batteries,140.7,99.7,99.7,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [101]:
generator_features_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 32050 entries, 0 to 32049
Data columns (total 34 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   State                                        32049 non-null  str    
 1   County                                       32049 non-null  str    
 2   Energy Source 1                              32049 non-null  str    
 3   Technology                                   32049 non-null  str    
 4   Time to Full Load                            32050 non-null  float64
 5   Summer Capacity (MW)                         31964 non-null  float64
 6   Winter Capacity (MW)                         31938 non-null  float64
 7   All Other                                    32050 non-null  int64  
 8   Batteries                                    32050 non-null  int64  
 9   Coal Integrated Gasification Combined Cycle  32050 non-null  int64  
 10  Conventio

In [108]:
non_energy = {
    "State",
    "County",
    "Energy Source 1",
    "Technology",
    "Time to Full Load",
    "Summer Capacity (MW)",
    "Winter Capacity (MW)"
}

energy_source = [
    energytech for energytech in generator_features_full.columns
    if energytech not in non_energy
]

group_by_stat = generator_features_full.groupby(["State","County"], as_index = False).agg(
    generator_count = ("County","size"),
    generator_summer_max = ("Summer Capacity (MW)","max"),
    generator_winter_max = ("Winter Capacity (MW)","max"),
    generator_avg_time_to_full = ("Time to Full Load","mean"),
    **{
        source:(source,"max")
        for source in energy_source
    }
)
group_by_stat.sample(5)

,State,County,generator_count,generator_summer_max,generator_winter_max,generator_avg_time_to_full,All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass
2098,VA,Goochland,1,5.0,5.0,140.700000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1813,TN,Hawkins,4,383.0,376.0,720.000000,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,AK,Aleutians East,14,1.1,1.1,28.671429,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
915,MI,Clare,2,2.0,2.0,100.350000,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1574,OK,Caddo,23,311.0,311.0,227.695652,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0


In [109]:
group_by_stat.to_csv("data/processed/EIA860_Generator_Cleaned.csv", index=False)

### EIA-861M Retail Sales of Electricity


In [63]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

Rows and columns: (10150, 24)


,Year,Month,State,Data Status,Thousand Dollars,Megawatthours,Count,Cents/kWh,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1,Thousand Dollars.2,Megawatthours.2,Count.2,Cents/kWh.2,Thousand Dollars.3,Megawatthours.3,Count.3,Cents/kWh.3,Thousand Dollars.4,Megawatthours.4,Count.4,Cents/kWh.4
593,2025,8.0,NM,Preliminary,129725.25,801132.37,920284.0,16.19,115923.78,946912.5,147252.0,12.24,84500.51,1315922.6,9190.0,6.42,0.00,0.00,0.0,0.00,330149.53,3063967.5,1076726.0,10.78
7983,2013,7.0,NC,Final,632253.06,5623270.20,4270143.0,11.24,405317.09,4462245.8,655098.0,9.08,175783.17,2523422.8,10361.0,6.97,51.12,651.33,1.0,7.85,1213404.40,12609590.0,4935603.0,9.62
6249,2016,5.0,NC,Final,419236.82,3714716.60,4414400.0,11.29,325834.87,3817997.8,671691.0,8.53,143578.56,2351299.3,10114.0,6.11,38.61,480.35,1.0,8.04,888688.86,9884494.0,5096206.0,8.99


In [64]:
sales.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,10150.0,18.0,2025.0,612.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,10149.0,NaN,NaN,NaN,6.41206,3.442597,1.0,3.0,6.0,9.0,12.0
State,10149,51,AK,199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Status,10149,2,Final,9180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thousand Dollars,10149.0,NaN,NaN,NaN,322406.257117,374363.877149,13064.0,78291.42,218232.0,404119.32,3266301.5
Megawatthours,10149.0,NaN,NaN,NaN,2358136.574637,2557344.034097,104574.0,595182.0,1706973.5,3135968.7,22481421.0
Count,10149.0,NaN,NaN,NaN,2631403.392847,2731937.969772,226548.0,802281.0,1971280.0,3106777.0,14863374.0
Cents/kWh,10149.0,NaN,NaN,NaN,14.26151,5.25453,6.73,11.08,12.7,15.63,52.72
Thousand Dollars.1,10149.0,NaN,NaN,NaN,251960.525962,316264.755862,20778.0,66788.65,166221.67,313892.57,3327766.4
Megawatthours.1,10149.0,NaN,NaN,NaN,2241268.764867,2416984.780473,126343.74,642345.04,1623061.3,2826659.0,17843545.0


In [65]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 10150 entries, 0 to 10149
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Year                10150 non-null  object 
 1   Month               10149 non-null  float64
 2   State               10149 non-null  str    
 3   Data Status         10149 non-null  str    
 4   Thousand Dollars    10149 non-null  float64
 5   Megawatthours       10149 non-null  float64
 6   Count               10149 non-null  float64
 7   Cents/kWh           10149 non-null  float64
 8   Thousand Dollars.1  10149 non-null  float64
 9   Megawatthours.1     10149 non-null  float64
 10  Count.1             10149 non-null  float64
 11  Cents/kWh.1         10149 non-null  float64
 12  Thousand Dollars.2  10149 non-null  float64
 13  Megawatthours.2     10149 non-null  float64
 14  Count.2             10149 non-null  float64
 15  Cents/kWh.2         10149 non-null  float64
 16  Thousand Dollar

In [66]:
sales['Year'] = sales['Year'].astype(str)

In [67]:
# Looking at 2025 only and only .1 since that is Commercial
# EIA classifies data centers as commercial customers
sales_2025 = sales[sales['Year']=='2025'].copy()
sales_2025 = sales_2025[['Year','Month','State','Thousand Dollars.1','Megawatthours.1','Count.1','Cents/kWh.1']]
sales_2025.sample(5)

,Year,Month,State,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
645,2025,7.0,NV,141293.51,1393355.2,179071.0,10.14
836,2025,3.0,MD,294301.27,2127976.3,262084.0,13.83
636,2025,7.0,MO,375137.94,3073463.3,407687.0,12.21
825,2025,3.0,FL,870436.59,7384092.0,1319993.0,11.79
652,2025,7.0,SC,309612.43,2821158.0,414961.0,10.97


In [110]:
# Group by State, Summer/Winter AVG
# Summer/Winter to match generator dataset capacity values
sales_2025['Season'] = sales_2025['Month'].apply(
    lambda x: 'summer' if 4 <= x <= 9 else 'winter'
)
grouped_sales = sales_2025.groupby(by=['State','Season'], as_index=False).mean(numeric_only=True).sort_values(by='State', ascending=True).drop(columns=['Month'])
grouped_sales.head(10)

,State,Season,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1
0,AK,summer,4.818776e+04,2.120679e+05,5.840083e+04,22.718333
1,AK,winter,5.055212e+04,2.303088e+05,5.806733e+04,21.958333
2,AL,summer,2.980218e+05,2.057032e+06,3.959620e+05,14.486667
3,AL,winter,2.590876e+05,1.779264e+06,3.959318e+05,14.563333
4,AR,summer,1.166311e+05,1.070972e+06,2.092887e+05,10.906667
5,AR,winter,9.688299e+04,9.132678e+05,2.091840e+05,10.611667
6,AZ,summer,4.453811e+05,3.455614e+06,3.505910e+05,12.855000
7,AZ,winter,3.346111e+05,2.801567e+06,3.505335e+05,11.940000
8,CA,summer,2.682709e+06,9.741230e+06,1.776620e+06,27.353333
9,CA,winter,2.304272e+06,9.178307e+06,1.766771e+06,25.031667


In [112]:
# Pivot and Flatten

pivot = (
    grouped_sales.pivot(
        index = 'State',
        columns = 'Season',
        values = [
            'Thousand Dollars.1',
            'Megawatthours.1',
            'Count.1',
            'Cents/kWh.1'
            ]
    )
)

pivot.columns = [
    f'{season}_{metric}'
    for metric, season in pivot.columns
    ]

pivot_sales = pivot.reset_index()
pivot_sales.head(20)


,State,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1
0,AK,4.818776e+04,5.055212e+04,2.120679e+05,2.303088e+05,5.840083e+04,5.806733e+04,22.718333,21.958333
1,AL,2.980218e+05,2.590876e+05,2.057032e+06,1.779264e+06,3.959620e+05,3.959318e+05,14.486667,14.563333
2,AR,1.166311e+05,9.688299e+04,1.070972e+06,9.132678e+05,2.092887e+05,2.091840e+05,10.906667,10.611667
3,AZ,4.453811e+05,3.346111e+05,3.455614e+06,2.801567e+06,3.505910e+05,3.505335e+05,12.855000,11.940000
4,CA,2.682709e+06,2.304272e+06,9.741230e+06,9.178307e+06,1.776620e+06,1.766771e+06,27.353333,25.031667
5,CO,2.322718e+05,2.006505e+05,1.779203e+06,1.693516e+06,4.017005e+05,4.015098e+05,13.028333,11.851667
6,CT,2.252811e+05,2.169308e+05,9.864365e+05,9.270143e+05,1.563380e+05,1.565435e+05,22.905000,23.406667
7,DC,1.251229e+05,1.205704e+05,6.262489e+05,5.772834e+05,2.759117e+04,2.761783e+04,19.915000,20.900000
8,DE,4.709125e+04,4.300572e+04,3.754953e+05,3.374110e+05,6.092983e+04,6.083633e+04,12.545000,12.751667
9,FL,1.039294e+06,8.833459e+05,9.121830e+06,7.636917e+06,1.324915e+06,1.324531e+06,11.398333,11.565000


In [113]:
pivot_sales.to_csv("data/processed/EIA861M_Sales_cleaned.csv", index=False)

### EIA-930 Balance Data

In [ ]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

In [ ]:
balance.describe(include="all").T

### EIA-930 Subregion Data

In [ ]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

In [ ]:
subregion.describe(include="all").T

## Save cleaned data

Save the cleaned data centers, including the `is_crypto` label, to `data/processed/`. This file is the input to `03-supervised-learning.ipynb`.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")

### Join Energy on cleaned fractacker data

In [122]:
data_centers_prep = data_centers.copy().rename(columns={"state":"State", "county":"County"})
data_center_energy_merge = data_centers_prep.merge(pivot_sales, on="State", how="left")
data_center_with_energy = data_center_energy_merge.merge(generator_features_full, on=["State","County"], how="left")
data_center_with_energy = data_center_with_energy.rename(columns={"State":"state","County":"county"}).reset_index()
data_center_with_energy.sample(5)

,index,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind,buildings_low,buildings_high,air,hybrid_air_water,water,closed_loop,fans,open_loop,crypto_miles,crypto_name,crypto_company,crypto_status,crypto_from_distance,crypto_from_address,crypto_from_dc_dataset,is_crypto,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass
26709,26709,3968933,1877fff8-5eae-46a0-85a2-f8d721d54232,1769137644043-9598,Switch Las Vegas CORE campus,7135 s decatur blvd,Las Vegas,NV,89118.0,Clark,36.060300,-115.211890,Operating,High,Google Maps and other sources,NaN,Switch,NaN,495,495.0,495.0,Hyperscale (100-999 MW),5,NaN,NaN,NaN,NaN,NaN,NaN,3500000.0,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://www.datacenterdynamics.com/en/news/swi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.769040e+12,1.769040e+12,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,226.273185,NaN,NaN,NaN,False,False,False,0,122040.948333,98712.205000,1.289148e+06,1.069415e+06,178522.333333,178582.666667,9.428333,9.248333,SUN,Solar Photovoltaic,140.7,26.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
20813,20813,3968653,22004fd5-a44f-42a6-a683-b378740c4d6d,256,Karis Critical Data Center,e 101st ave,Crown Point,IN,46307.0,Lake,41.435210,-87.309900,Proposed,Medium,Google Maps and other sources,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181.0,$900 million,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Media Monitoring,https://nwindianabusiness.com/community/busine...,https://www.govtech.com/artificial-intelligenc...,NaN,NaN,NaN,NaN,NaN,NaN,1.756858e+12,1.756858e+12,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,63.250749,NaN,NaN,NaN,False,False,False,0,309199.973333,280859.316667,2.205490e+06,2.046696e+06,375566.333333,375833.500000,14.026667,13.738333,SUN,Solar Photovoltaic,140.7,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42940,42940,3969706,ecd38bff-df0a-46c2-92e9-92db159015db,1014,Amazon AWS IAD83,9001 hornbaker rd,Manassas,VA,20109.0,Prince William,38.762554,-77.533850,Proposed,High,Other sources,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,1,NaN,NaN,NaN,NaN,NaN,NaN,871518.0,40.0,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other,PEC,https://egcss.pwcgov.org/SelfService#/plan/a0b...,https://pwc.publicaccessnow.com/PropertyDetail..

In [125]:
data_center_with_energy.to_csv("data/processed/fractracker_data_centers_with_eia_cleaned.csv", index=False)
test = pd.read_csv("data/processed/fractracker_data_centers_with_eia_cleaned.csv", header=0)

C:\Users\Stormblessed\AppData\Local\Temp\ipykernel_36240\611043081.py:2: DtypeWarning: Columns (25: number_of_generators, 26: number_of_buildings, 32: expected_date_online, 36: nda, 38: community_group_website_2, 48: info_source_6, 49: info_source_7, 50: info_source_8, 53: delete_dup, 77: crypto_name, 78: crypto_company, 79: crypto_status) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("data/processed/fractracker_data_centers_with_eia_cleaned.csv", header=0)


In [126]:
test.head(5)

,index,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind,buildings_low,buildings_high,air,hybrid_air_water,water,closed_loop,fans,open_loop,crypto_miles,crypto_name,crypto_company,crypto_status,crypto_from_distance,crypto_from_address,crypto_from_dc_dataset,is_crypto,summer_Thousand Dollars.1,winter_Thousand Dollars.1,summer_Megawatthours.1,winter_Megawatthours.1,summer_Count.1,winter_Count.1,summer_Cents/kWh.1,winter_Cents/kWh.1,Energy Source 1,Technology,Time to Full Load,Summer Capacity (MW),Winter Capacity (MW),All Other,Batteries,Coal Integrated Gasification Combined Cycle,Conventional Hydroelectric,Conventional Steam Coal,Flywheels,Geothermal,Hydroelectric Pumped Storage,Landfill Gas,Municipal Solid Waste,Natural Gas Fired Combined Cycle,Natural Gas Fired Combustion Turbine,Natural Gas Internal Combustion Engine,Natural Gas Steam Turbine,Natural Gas with Compressed Air Storage,Nuclear,Offshore Wind Turbine,Onshore Wind Turbine,Other Gases,Other Natural Gas,Other Waste Biomass,Petroleum Coke,Petroleum Liquids,Solar Photovoltaic,Solar Thermal with Energy Storage,Solar Thermal without Energy Storage,Wood/Wood Waste Biomass
0,0,3968264,f4107c30-66df-4c1e-8022-28e6c8fe2b5d,1779305646902-5353,Stak Energy Data Center,dalton hwy 26 miles s of deadhorse,North Slope Borough,AK,99734.0,North Slope,69.90071,-148.81477,Proposed,Medium,Google Maps and other sources,AI/Cloud Computing,Stak,NaN,3000,3000.0,3000.0,"Mega campus (>1,000 MW)",6,Natural gas,Yes,NaN,NaN,Air,NaN,NaN,715.0,$500 million,2028.0,Unknown,NaN,Emerging Concern,NaN,NaN,NaN,NaN,NaN,KE/JH,Media Monitoring,https://www.datacenterdynamics.com/en/news/sta...,https://www.cbc.ca/news/canada/north/yukon-ala...,https://alaskapublic.org/news/economy/energy/2...,https://alaskabeacon.com/2026/07/06/data-cente...,https://aws.state.ak.us/OnlinePublicNotices/No...,NaN,NaN,NaN,1.779235e+12,1.783469e+12,NaN,0,0,0,0,0,0,0,0,1,0,0,0,0,0,NaN,NaN,1,0,0,0,0,0,12.953958,NaN,NaN,NaN,False,False,False,0,48187.756667,50552.12,212067.873333,230308.786667,58400.833333,58067.333333,22.718333,21.958333,NG,Natural Gas Fired Combustion Turbine,10.0,4.8,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,3968264,f4107c30-66df-4c1e-8022-28e6c8fe2b5d,1779305646902-5353,Stak Energy Data Center,dalton hwy 26 miles s of deadhorse,North Slope Borough,AK,99734.0,North Slope,69.90071,-148.81477,Proposed,Medium,Google Maps and other sources,AI/Cloud Computing,Stak,NaN,3000,3000.0,3000.0,"Mega campus (>1,000 MW)",6,Natural gas,Yes,NaN,NaN,Air,NaN,NaN,715.0,$500 million,2028.0,Unknown,NaN,Emerging Concern,NaN,NaN,NaN,NaN,NaN,KE/JH,Media Monitoring,https://www.datacenterdynamics.com/en/news/sta...,https://www.cbc.ca/news/canada/north/yukon-ala...,https://alaskapublic.org/news/economy/energy/2...,https://alaskabeacon.com/2026/07/06/data-cente...,https://aws.state.ak.us/OnlinePublicNotices/No...,NaN,NaN,NaN,1.779235e+12,1.783469e+12,NaN,0,0,0,0,0,0,0,0,1,0,0,0,0,0,NaN,NaN,1,0,0,0,0,0,12.953958,NaN,NaN,NaN,False,False,False,0,48187.756667,50552.12,212067.873333,230308.786667,58400.833333,58067.333333,22.718333,21.958333,NG,Natural Gas Fired Combustion Turbine,10.0,5.0